In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import torch

In [2]:
data_path = "/home/exouser/fireAirqaulity/training_data/X_train_1.csv"
column_names = ["u10", "v10","tp","t2m","sp","d2m","frp","cur_pm25","fut_pm25","near_fires"]

df = pd.read_csv(data_path, names=column_names, header=None)
frp_threshold = 0.01
df = df[df["cur_pm25"] > 0]
df = df[df["fut_pm25"] > 0]
df_clipped = df[df["frp"] > frp_threshold].dropna()
df = df_clipped
df

,u10,v10,tp,t2m,sp,d2m,frp,cur_pm25,fut_pm25,near_fires
19790,-1.435523,0.350677,1.322508e-07,303.314911,86800.0,277.471436,0.065685,0.649051,1.649051,4.0
20262,0.784682,2.299662,0.000000e+00,307.436951,100680.0,287.834839,0.013026,6.183359,9.183359,22.0
21230,0.317669,3.437999,0.000000e+00,307.537872,100690.0,285.999786,0.025062,9.183359,3.483359,22.0
22287,1.151041,1.339413,0.000000e+00,309.256683,101250.0,285.124786,0.030610,1.219918,0.419918,14.0
22289,-0.598959,5.589413,4.676475e-07,309.381683,100100.0,286.437286,0.013110,2.358887,2.558887,0.0
...,...,...,...,...,...,...,...,...,...,...
707815,2.866973,2.018118,3.785940e-05,304.095123,93720.0,281.640320,0.615279,1.042292,5.042292,4.0
708767,-1.453134,0.445147,0.000000e+00,298.882751,91930.0,279.877563,3.692392,145.357625,110.357625,6.0
709726,-2.131605,0.215801,0.000000e+00,301.907806,91860.0,278.304413,3.655234,110.357625,94.357625,8.0
710685,-2.017550,0.669683,0.000000e+00,304.350922,91810.0,276.656403,3.730983,94.357625,104.357625,11.0


In [3]:
class mlp(torch.nn.Module):
    def __init__(self,inputsize, depth, width):
        super().__init__()
        self.inp = torch.nn.Linear(inputsize, width)
        self.blocks = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Linear(width, width),
                torch.nn.LeakyReLU(),
            ) for _ in range(depth)
        ])
        self.out = torch.nn.Linear(width, 1)

    def forward(self, x):
        x = self.inp(x)
        for block in self.blocks:
            x = block(x)
        return self.out(x)


In [4]:
data_y = df["fut_pm25"].to_numpy()
data_x = df.drop(columns="fut_pm25").to_numpy()

x_mean = data_x.mean(axis=0)
x_std = data_x.std(axis=0)
data_x_norm = (data_x - x_mean) / x_std
y_mean = data_y.mean(axis=0)
y_std = data_y.std(axis=0)
data_y_norm = (data_y - y_mean) / y_std

In [ ]:
# data_x_norm = data_x
# data_y_norm = data_y

In [5]:
model = mlp(inputsize=9,depth=80,width=80)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
L = torch.nn.MSELoss()

n_epochs = 10
loss_list = []
for epoch in range(n_epochs):
    for x,y in zip(data_x_norm,data_y_norm):
        x_tensor = torch.tensor(x,dtype=torch.float32)
        y_tensor = torch.tensor(y,dtype=torch.float32) 
        optimizer.zero_grad()
        y_pred = model(x_tensor)
        train_loss = L(y_tensor,y_pred[0])
        print(f"Training Loss: {train_loss.item()}")
        loss_list.append(train_loss.item())
        train_loss.backward() 
        optimizer.zero_grad()


Training Loss: 0.2791211009025574
Training Loss: 0.051211271435022354
Training Loss: 0.11874867975711823
Training Loss: 0.4243561029434204
Training Loss: 0.19113697111606598
Training Loss: 0.2791211009025574
Training Loss: 0.36900773644447327
Training Loss: 0.2791211009025574
Training Loss: 0.2791211009025574
Training Loss: 0.29211530089378357
Training Loss: 0.35706019401550293
Training Loss: 0.44904083013534546
Training Loss: 0.2791211009025574
Training Loss: 0.01955702342092991
Training Loss: 0.34025153517723083
Training Loss: 0.4762885570526123
Training Loss: 0.1938810646533966
Training Loss: 0.32866722345352173
Training Loss: 0.4625643491744995
Training Loss: 0.29211530089378357
Training Loss: 0.3463095426559448
Training Loss: 0.26337477564811707
Training Loss: 0.4104125201702118
Training Loss: 0.390934556722641
Training Loss: 0.3231344223022461
Training Loss: 0.20530498027801514
Training Loss: 0.4251368045806885
Training Loss: 0.29211530089378357
Training Loss: 0.01854356378316879

In [ ]:
plt.plot(loss_list)
plt.show()

In [ ]:
model.eval() 
outputs = []
with torch.no_grad():
    for x in data_x_norm:
        x_tensor = torch.tensor(x,dtype=torch.float32)
        outputs.append(model(x_tensor))
        

In [ ]:
plt.plot(outputs)
plt.plot(data_y_norm)

### OVERTRAINING TEST

In [ ]:
model = mlp(inputsize=9,depth=80,width=80)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
L = torch.nn.MSELoss()

data_x_norm_subset = data_x_norm[1:2]
data_y_norm_subset = data_y_norm[1:2]

n_epochs = 1000
loss_list = []
for epoch in range(n_epochs):
    for x,y in zip(data_x_norm_subset,data_y_norm_subset):
        x_tensor = torch.tensor(x,dtype=torch.float32)
        y_tensor = torch.tensor(y,dtype=torch.float32) 
        optimizer.zero_grad()
        y_pred = model(x_tensor)
        train_loss = L(y_tensor,y_pred[0])
        print(f"Training Loss: {train_loss.item()}")
        loss_list.append(train_loss.item())
        train_loss.backward() 

In [ ]:
model.eval() 
outputs = []
with torch.no_grad():
    x_tensor = torch.tensor(data_x_norm[1],dtype=torch.float32)
    print(model(x_tensor))
    print(data_y_norm_subset)

In [ ]:
plt.plot(loss_list)
plt.show()